In [174]:
from helper_functions import *

In [175]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from tqdm import tqdm

In [176]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [177]:
states = []
rewards = []
for i in tqdm(range(len(months))):
    state_i, reward_i = get_state_reward(i)
    states.append(state_i)
    rewards.append(reward_i)

100%|█████████████████████████████████████████████████████████████████████████████████| 311/311 [01:23<00:00,  3.71it/s]


In [178]:
class MLPEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [179]:
class CrossStockAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.WQ = nn.Linear(d_model, d_model)
        self.WK = nn.Linear(d_model, d_model)
        self.WV = nn.Linear(d_model, d_model)

        self.ws = nn.Linear(d_model, 1)

    def forward(self, r):
        Q = self.WQ(r)
        K = self.WK(r)
        V = self.WV(r)

        scores = torch.matmul(Q, K.T) / math.sqrt(Q.shape[1])
        alpha = F.softmax(scores, dim=1)

        A = torch.matmul(alpha, V)

        s = self.ws(A).squeeze(-1)

        return s

In [180]:
def construct_portfolio_soft(s):
    long_weights  = torch.softmax(s, dim=0)
    short_weights = torch.softmax(-s, dim=0)
    b = long_weights - short_weights
    b = b / (b.abs().sum() + 1e-8)   # <-- normalize so weights sum to ±1
    return b

In [181]:
class TradingModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, d_model=16):
        super().__init__()

        self.encoder = MLPEncoder(input_dim, hidden_dim, d_model)
        self.attention = CrossStockAttention(d_model)

    def forward(self, state_i, reward_i):

        x = torch.tensor(
            [s.to_numpy() if hasattr(s, "to_numpy") else s for s in state_i],
            dtype=torch.float32
        )

        returns = torch.tensor(reward_i, dtype=torch.float32)

        r = self.encoder(x)
        s = self.attention(r)

        b = construct_portfolio_soft(s)

        R_t = torch.sum(b * returns)

        return r, s, b, R_t

In [182]:
def compute_loss(R_list):
    R = torch.stack(R_list)
    mu = R.mean()
    sigma = R.std() + 1e-6
    
    sharpe = mu / sigma
    mean_return = mu
    
    # Penalize drawdown
    cumulative = torch.cumprod(1 + R, dim=0)
    peak = torch.cummax(cumulative, dim=0).values
    drawdown = ((peak - cumulative) / (peak + 1e-8)).max()
    
    loss = -0.5 * sharpe - 0.5 * mean_return * 12 + 0.2 * drawdown
    return loss

In [183]:
model = TradingModel(len(features))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

def train(model, optimizer, scheduler, all_states=states, all_rewards=rewards, epochs=50):
    window = 12
    T = len(all_states) - 24  # hold out last 24 months

    for epoch in range(epochs):
        sharpe_list, return_list = [], []

        for start in range(T - window):
            optimizer.zero_grad()

            R_list = []
            for t in range(start, start + window):
                _, _, _, R_t = model(all_states[t], all_rewards[t])
                R_list.append(R_t)

            loss = compute_loss(R_list)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            R = torch.stack(R_list)
            sharpe_list.append((R.mean() / (R.std() + 1e-6)).item())
            return_list.append(R.sum().item())

        scheduler.step()

        print(f"Epoch {epoch:03d} | Sharpe: {sum(sharpe_list)/len(sharpe_list):.4f} | "
              f"Avg Return: {sum(return_list)/len(return_list):.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.6f}")

In [184]:
set_seed(42)
model = TradingModel(len(features))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

train(model, optimizer, scheduler, epochs=50)

Epoch 000 | Sharpe: 0.4814 | Avg Return: 0.0792 | LR: 0.000999
Epoch 001 | Sharpe: 0.5757 | Avg Return: 0.1077 | LR: 0.000996
Epoch 002 | Sharpe: 0.6430 | Avg Return: 0.1153 | LR: 0.000991
Epoch 003 | Sharpe: 0.6476 | Avg Return: 0.1152 | LR: 0.000984
Epoch 004 | Sharpe: 0.7584 | Avg Return: 0.1384 | LR: 0.000976
Epoch 005 | Sharpe: 0.7894 | Avg Return: 0.1245 | LR: 0.000965
Epoch 006 | Sharpe: 0.8285 | Avg Return: 0.1371 | LR: 0.000953
Epoch 007 | Sharpe: 0.9192 | Avg Return: 0.1482 | LR: 0.000939
Epoch 008 | Sharpe: 0.8760 | Avg Return: 0.1384 | LR: 0.000923
Epoch 009 | Sharpe: 0.9555 | Avg Return: 0.1494 | LR: 0.000905
Epoch 010 | Sharpe: 0.9938 | Avg Return: 0.1499 | LR: 0.000886
Epoch 011 | Sharpe: 1.0688 | Avg Return: 0.1578 | LR: 0.000866
Epoch 012 | Sharpe: 1.0349 | Avg Return: 0.1572 | LR: 0.000844
Epoch 013 | Sharpe: 1.1389 | Avg Return: 0.1643 | LR: 0.000821
Epoch 014 | Sharpe: 1.1277 | Avg Return: 0.1614 | LR: 0.000796
Epoch 015 | Sharpe: 1.2430 | Avg Return: 0.1714 | LR: 0

In [208]:
test_states = states[-12:]
test_rewards = rewards[-12:]

R_list = []
for t in range(12):
    _, _, _, R_t = model(test_states[t], test_rewards[t])
    R_list.append(R_t)

R = torch.stack(R_list)

print("Test Sharpe:", (R.mean() / R.std()).item())

Test Sharpe: 0.05755201727151871


In [209]:
print("Total Return:", R.sum().item())
print("Sharpe:", (R.mean() / R.std()).item())
print("Final Equity:", torch.cumprod(1 + R, dim=0)[-1].item())

Total Return: 0.0055657047778368
Sharpe: 0.05755201727151871
Final Equity: 1.0052211284637451
